### Etapa 1: Ingestão e Blindagem de Dados

Nesta célula, nosso "garçom" (biblioteca `requests`) vai à internet buscar o formato JSON contendo o histórico do Dólar dos últimos 15 dias.

**Proteção Cloud:** Como rodaremos este script na nuvem, a AwesomeAPI pode bloquear nosso acesso achando que somos um ataque automatizado. Para evitar isso, instruímos a IA a enviar um "User-Agent", disfarçando nosso robô como um navegador Google Chrome comum.

**🤖 PROMPT ENVIADO PARA A IA:**
> "Crie um script em Python que acesse a AwesomeAPI para buscar a cotação do dólar dos últimos 15 dias em formato JSON. Muito importante: passe um cabeçalho (header) de 'User-Agent' simulando o navegador Google Chrome para evitar bloqueios de segurança (erro 403) no Google Colab. Percorra os dados, extraia a data convertendo o 'timestamp' numérico para o formato Ano-Mês-Dia e extraia o valor. Guarde essas duas informações blindadas dentro de uma Tupla e adicione em uma lista chamada historico_dolar."

In [ ]:
import requests
from datetime import datetime

url_api = "https://economia.awesomeapi.com.br/json/daily/USD-BRL/15"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

resposta = requests.get(url_api, headers=headers)
dados_json = resposta.json()

historico_dolar = []

for dia in dados_json:
    ts_inteiro = int(dia["timestamp"])
    data = datetime.fromtimestamp(ts_inteiro).strftime("%Y-%m-%d")
    valor = float(dia["bid"])
    
    tupla_diaria = (data, valor)
    historico_dolar.append(tupla_diaria)
    
print("Coleta concluída! Últimos 3 dias carregados:")
print(historico_dolar[:3])

### Etapa 2: Transformação de Dados via Lookup Table

Para evitar um código espaguete cheio de `if/elif/else`, utilizamos uma Matriz (Listas dentro de Listas). Cruzando o Eixo Y (Volatilidade) com o Eixo X (Tendência), a aplicação acessa a decisão de compra de forma instantânea através dos índices!

**🤖 PROMPT ENVIADO PARA A IA:**
> "Refatore a lógica de decisão de compra cambial baseada na variação do dia. Não utilize múltiplos blocos if/elif/else para a decisão final. Crie uma Matriz Multidimensional (Lookup Table) onde a Linha representa a Volatilidade (Baixa/Alta) e a Coluna representa a Tendência (Queda/Estável/Alta). Calcule os índices matematicamente e consulte a matriz diretamente para obter o status estratégico de cada dia."

In [ ]:
matriz_decisao = [
    ["✅ Comprar", "⏸️ Manter", "⏳ Aguardar Queda"],
    ["⚠️ Risco: Comprar", "⚠️ Risco: Manter", "🛑 Paralisar"]
]

relatorio_analitico = []

for i in range(len(historico_dolar)-2,-1,-1):
    hoje = historico_dolar[i][1]
    ontem = historico_dolar[i+1][1]
    variacao = hoje - ontem
    
    tendencia_idx = 0 if variacao < -0.02 else (2 if variacao > 0.02 else 1)
    vol_idx = 1 if abs(variacao) > 0.06 else 0
    
    decisao = matriz_decisao[vol_idx][tendencia_idx]
    
    relatorio_analitico.append({
        "Data": historico_dolar[i][0],
        "Cotacao_USD": hoje,
        "Variacao_Dia": round(variacao,4),
        "Acao_Estrategica": decisao
    })
    
print("Lógica de Matriz O(1) processada com sucesso!")

### Etapa 3: Visualização de Dados (Data Viz)

Como estamos em um arquivo iterativo `.ipynb`, podemos renderizar imagens nativamente! Utilizamos o `matplotlib` para plotar um gráfico de linhas dinâmico, evidenciando a tendência da cotação.

**🤖 PROMPT ENVIADO PARA A IA:**
> "Crie uma célula de código importando a biblioteca matplotlib. Extraia as datas e os valores do nosso dicionário relatorio_analitico e construa um gráfico de linhas (lineplot). Formate o visual de forma profissional: adicione grid tracejado, rotacione o eixo X em 45 graus para facilitar a leitura das datas, pinte a linha do dólar de verde e garanta que a prancheta seja renderizada nativamente abaixo da célula."

In [ ]:
import matplotlib.pyplot as plt

datas = [item["Data"] for item in relatorio_analitico]
valores_usd = [item["Cotacao_USD"] for item in relatorio_analitico]

fig,eixo1 = plt.subplots(figsize=(10,5))

eixo1.plot(datas, valores_usd, color="green", marker="o", linewidth=2, label="Dólar (BRL)")
eixo1.set_xlabel("Timeline", fontweight="bold")
eixo1.set_ylabel("Preço USD", color="green", fontweight="bold")
eixo1.tick_params(axis="x", rotation=45)

plt.title("Monitoramento Cambial - TechSolutions", fontsize=14, pad=15)
plt.grid(True,linestyle="--",alpha=0.6)
plt.tight_layout()

plt.show()